# Projeto 1 - T319 (1S2026)

### Instruções:

1. Quando você terminar os exercícios do projeto, vá até o menu do Jupyter ou Colab e selecione a opção para fazer download do notebook.
    * Os notebooks tem extensão .ipynb.
    * Este deve ser o arquivo que você irá entregar.
    * No Colab vá até a opção **File** -> **Download .ipynb**.
2. Após o download do notebook, vá até a aba de tarefas do MS Teams, localize a tarefa referente a este projeto e faça o upload do seu notebook. Veja que há uma opção de anexar arquivos à tarefa.
3. Atente-se ao prazo de entrega definido na tarefa do MS Teams. Entregas fora do prazo não serão aceitas.
4. **O projeto pode ser resolvido em grupos de no MÁXIMO 3 alunos**.
5. Todas as questões têm o mesmo peso.
6. Não se esqueça de colocar seu(s) nome(s) e número(s) de matrícula no campo abaixo.
7. Você pode consultar todo o material de aula.
8. A interpretação faz parte do projeto. Leia o enunciado de cada questão atentamente!
9. Boa sorte!

---



**Nomes e matrículas**:

1.

2.

3.

# 1) **Exercício sobre a escolha do passo de aprendizagem**

1. Execute a célula de código abaixo para importar as bibliotecas e definir algumas funções necessárias para o treinamento de um modelo de regressão linear.

**DICAS**

+ A função `gradientDescent` implementa a versão **estocástica** do gradiente descendente.
+ Note que a função `gradientDescent` utiliza **decaimento temporal** do passo de aprendizagem para tornar o aprendizado do algoritmo mais comportado.

In [ ]:
# Import all necessary libraries.
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Reseta os gerados de sequências pseudo-aleatórias.
seed = 3
np.random.seed(seed)
random.seed(seed)

def calculateErrorSurface(x, y):
    """Generate data points for plotting the error surface."""

    # Retrieve number of examples.
    N = len(y)

    # Generate values for parameter space.
    M = 200
    a0 = np.linspace(0.995, 1.005, M)
    a1 = np.linspace(0.495, 0.505, M)

    # Generate matrices with combinations between a0 and a1 values.
    A0, A1 = np.meshgrid(a0, a1)

    # Generate points for plotting the cost-function surface.
    J = np.zeros((M,M))
    for iter1 in range(0, M):
        for iter2 in range(0, M):
            # Hypothesis function (a second degree function).
            yhat = A0[iter1, iter2] + A1[iter1, iter2]*np.sign(np.sin(2 * np.pi * 2.5 * x))
            # Calculate the mean squared error (MSE) for each pair of values.
            J[iter1, iter2] = (1.0/N)*np.sum(np.square(y - yhat))

    return J, A0, A1

def timeBasedDecay(alpha_init, k, t):
    '''Decaimento temporal.'''
    return alpha_init / (1.0 + k*t)

def gradientDescent(X_train, y_train, X_test, y_test, n_epochs, alpha_init, k):
    '''
    Função que implementa a versão estocástica do gradiente descendente.
    Os parâmetros de entrada da função são:
    * X_train    - Matriz de atributos de treinamento
    * y_train    - vetor de rótulos de treinamento
    * X_test     - Matriz de atributos de teste
    * y_test     - vetor de rótulos de teste
    * n_epochs   - número máximo de épocas de treinamento
    * alpha_init - valor inicial do passo de aprendizagem
    * k          - taxa de decaimento da redução temporal do passo de aprendizagem

    Os valores retornados são:
    * a               : vetor de pesos correspondente à última iteração de atualização
    * a_min           : vetor de pesos correspondente ao menor erro de teste
    * Jgd             : vetor com os valores do erro de treinamento ao longo do treinamento
    * Jgd_test        : vetor com os valores do erro de teste ao longo do treinamento
    * a_hist          : matriz com o histórico dos valore do vetor de pesos
    * alpha_hist      : vetor com histórico dos valores do passo de aprendizagem
    * update_hist     : matriz com o histórico de vetores de atualização dos pesos
    * gradient_hist   : matriz com o histórico de vetores gradiente
    * iteration       : valor da última iteração
    '''

    # Reseta os geradores de sequências pseudo-aleatórias.
    np.random.seed(seed)
    random.seed(seed)

    # Number of training examples.
    N_train = len(y_train)

    # Number of test examples.
    N_test = len(y_test)

    # Reshape y to be a column vector.
    y_train = y_train.reshape(N_train,1)

    # Inicialização do vetor de pesos.
    a = np.array([-5.0, -4.0]).reshape(2, 1)

    # Create vector for parameter history.
    a_hist = np.zeros((2, n_epochs*N_train+1))
    # Initialize history vector.
    a_hist[:, 0] = a.reshape(2,)

    # Create vector to store eta history.
    alpha_hist = np.zeros((n_epochs*N_train))

    # Create array for storing training error values.
    Jgd = np.zeros(n_epochs*N_train+1)

    # Create array for storing test error values.
    Jgd_test = np.zeros(n_epochs*N_train+1)

    # Calcule o MSE com conjunto de treinamento para o conjunto de pesos inicial.
    Jgd[0] = (1.0/N_train)*np.sum(np.power(y_train - X_train.dot(a), 2))

    # Calcule o MSE com conjunto de teste para o conjunto de pesos inicial.
    Jgd_test[0] = (1.0/N_test)*np.sum(np.power((y_test - X_test.dot(a)), 2))

    # Cria arrays para armazenar vetores de atualização e gradiente.
    update_hist = np.zeros((2, n_epochs*N_train))
    gradient_hist = np.zeros((2, n_epochs*N_train))

    # Stocastic gradient-descent loop.
    iteration = 0
    min_test_error = float('inf')
    min_test_error_iteration = 0
    a_min = a
    # Época de treinamento, apresenta todas os exemplos de treinamento ao modelo.
    for epoch in range(n_epochs):

        # Shuffle the whole dataset before every epoch.
        shuffled_data_set_indexes = random.sample(range(0, N_train), N_train)

        # Iteração de treinamento, apenas um exemplo é apresentado ao modelo.
        for i in range(N_train):
            # Retrieve one pair of atribute vector and label.
            random_index = shuffled_data_set_indexes[i]
            xi = X_train[random_index:random_index+1]
            yi = y_train[random_index:random_index+1]

            # Decaimento temporal do passo de aprendizagem.
            alpha = timeBasedDecay(alpha_init, k, epoch*N_train + i)

            # Cálculo da estimativa do vetor gradiente com apenas uma amostra.
            gradient = -2.0*xi.T.dot(yi - xi.dot(a))
            update = alpha*gradient
            a = a - update

            # Armazena o histórico de valores.
            a_hist[:, epoch*N_train+i+1] = a.reshape(2,)
            alpha_hist[epoch*N_train+i] = alpha
            update_hist[:, epoch*N_train+i] = update.reshape(2,)
            gradient_hist[:, epoch*N_train+i] = gradient.reshape(2,)

            # Calcula o MSE com conjunto de treinamento por itereção de treinamento.
            Jgd[epoch*N_train+i+1] = (1.0/N_train)*np.sum(np.power((y_train - X_train.dot(a)), 2))

            # Calcula o MSE com conjunto de teste por itereção de treinamento.
            Jgd_test[epoch*N_train+i+1] = (1.0/N_test)*np.sum(np.power((y_test - X_test.dot(a)), 2))

            # Early-stopping.
            if Jgd_test[epoch*N_train+i+1] < min_test_error:
                min_test_error = Jgd_test[epoch*N_train+i+1]
                min_test_error_iteration = epoch*N_train+i+1
                a_min = a

            # Incrementa o contador de iterações.
            iteration = epoch*N_train+i

    return a, a_min, Jgd, Jgd_test, a_hist, alpha_hist, update_hist, gradient_hist, iteration

2. Execute a célula de código abaixo para criar o conjunto de dados que será usado neste exercício.

+ A função objetivo utilizada neste exercício é uma onda quadrada, dada por
$$
y = 1 + 0.5 \cdot \operatorname{sgn}\big(\sin(2\pi \ 2.5 x)\big),
$$
onde $\operatorname{sgn}(\cdot)$ representa a função sinal, que assume valores $+1$ ou $-1$. Essa função gera uma forma de onda periódica que alterna abruptamente entre dois níveis, característica típica de ondas quadradas.

+ A função hipótese que utilizaremos tem o formato
$$
\hat{y} = \hat{a}_0 + \hat{a}_1 \operatorname{sgn}\big(\sin(2\pi \cdot 2.5 x)\big),
$$
sendo o objetivo do algoritmo do gradiente descendente estocástico encontrar aproximações, $\hat{a}_0$ e $\hat{a}_1$, para os valores ideais, ${a}_0$ e ${a}_1$.

+ Para representarmos a função hipótese em formato matricial, i.e., $\mathbf{y} = \mathbf{X}\mathbf{a}$, precisamos criar a matriz de atributos concatenando os vetores dos atributos de bias (**vetor com valores iguais a 1**) e do atributo formado pela transformação do vetor $\mathbf{x}$: $\operatorname{sgn}(\sin(2\pi \cdot 2.5 \mathbf{x}))$.

**DICAS**

+ Na célula de código abaixo, o vetor do atributo de bias é concatenado ao vetor do atributo $\operatorname{sgn}(\sin(2\pi \cdot 2.5 \mathbf{x}))$, formando a matriz de atributos $\mathbf{X}$.
+ Essa concatenação é feita de forma manual, pois a implementação da versão estocástica do gradiente descendente fornecida acima não realiza esse procedimento automaticamente, diferentemente das classes da biblioteca Scikit-Learn.

In [ ]:
# Número de amostras
N = 1000

# Gera um vetor de atributo
x = np.linspace(0, 1, N, endpoint=False).reshape(N,1)

# Cria uma onda quadrada
y = 1 + 0.5*np.sign(np.sin(2 * np.pi * 2.5 * x))

# Ruído
w = np.sqrt(0.01)*np.random.randn(N, 1)

# Função observável
y_noisy = y + w

# Cria matriz de atributos
X = np.c_[np.ones((N,1)), np.sign(np.sin(2 * np.pi * 2.5 * x))]

# Plot
plt.plot(x, y_noisy, label='Função observável')
plt.plot(x, y, label='Função objetivo')
plt.xlabel('x', fontsize=14)
plt.ylabel('y', fontsize=14)
plt.grid()
plt.legend()
plt.show()

3. Analise a geração das amostras da função observável no item anterior e responda: qual é o menor erro (i.e., erro quadrático médio - EQM) possível com um regressor linear treinado com essas amostras?

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

4. Divida o conjunto total de amostras em conjuntos de treinamento e validação. O conjunto de treinamento deve conter 75% do total de amostras e o conjunto de validação os 25% restantes.

**DICAS**

+ Use a função `train_test_split` e a configure com os seguintes parâmetros `test_size=0.25` e `random_state=seed`. A função divide o conjunto original de amostras em dois subconjuntos, um para treinamento e outro para validação (i.e., para avaliar a capacidade de generalização do modelo). Veja o código abaixo.
```python
X_train, X_test, y_train, y_test = train_test_split(X, y_noisy, test_size=0.25, random_state=seed)
```
+ Para que o próximo item do exercício funcione, chame as matrizes de treinamento e de validação de `X_train` e `X_test`, respectivamente, e os vetores de rótulos de treinamento e de validação de `y_train` e `y_test`, respectivamente, como no exemplo acima.

In [ ]:
# Digite o código do exercício aqui.

5. Execute a célula de código abaixo e analise as figuras.

A célula abaixo treina o modelo de regressão usando a função `gradientDescent` com os seguintes valores:
+ **taxa de decaimento ($k$)**: 0.1, 0.05, e 0.01.
+ **passo de aprendizagem ($\alpha$)**: 0.495, 0.25, 0.125, 0.0625 e 0.03125.

Cada figura mostra o erro de treinamento em função das iterações de treinamento para um valor específico da taxa de decaimento ($k$) e vários valores para o passo de aprendizagem ($\alpha$). O valor de taxa de decaimento ($k$) é mostrado no título (i.e., topo) da figura, enquanto os diferentes valores de passo de aprendizagem ($\alpha$) são mostrados com cores diferentes na legenda de cada figura.

**DICA**:

+ Lembrem-se que o menor valor do EQM tende ao valor da variância do ruído adicionado às amostras da função objetivo quando encontra-se os valores ótimos dos pesos.

In [ ]:
# Número de épocas.
n_epochs = 2
# Lista de taxas de decaimento.
k_list = [0.025, 0.175, 0.325] #0.225
# Lista de passos de aprendizagem.
alpha_list = [0.25, 0.2, 0.15, 0.1]

# Lista para armazenar os erros das combinações de taxa de decaimento e passo de aprendizagem.
error = []
for k in k_list:
    error_hist = []
    for alpha in alpha_list:
        a, a_min, Jgd, Jgd_test, a_hist, alpha_hist, update_hist, gradient_hist, iteration = gradientDescent(X_train, y_train, X_test, y_test, n_epochs, alpha_init=alpha, k=k)
        error_hist.append(Jgd_test)
    error.append(error_hist)

# Visualização do erro durante o treinamento de cada passo de aprendizagem.
plt.figure(figsize=(25,7))
for i in range(len(k_list)):
    plt.subplot(1, 3, i+1)
    plt.title('k = '+str(k_list[i]))
    for j in range(len(alpha_list)):
        plt.plot(np.arange(error[i][j].shape[0]), error[i][j], label=('alpha = '+f'{alpha_list[j]}'))
        plt.yscale('log')
    plt.xlabel('Iterações')
    plt.ylabel('EQM')
    plt.legend()
    plt.grid()
    plt.ylim([0.0095, 0.02])
plt.show()

6. Analise as figuras do item anterior e responda: Quais são os valores ideais para a taxa de decaimento ($k$) e o passo de aprendizagem ($\alpha$)? (**Justifique sua resposta**).

**DICA**

+ A ideia é que o aprendizado seja rápido, ou seja, convirja rapidamente (erro praticamente constante), mas sem muita oscilação no erro.

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

7. De posse dos valores ideais para a taxa de decaimento ($k$) e passo de aprendizagem ($\alpha$), treine novamente o modelo com estes valores e imprima os erros quadráticos médios (EQMs) obtidos para os conjuntos de treinamento de validação e o valor dos pesos $\hat{a}_0$ e $\hat{a}_1$ correspondentes à última iteração de atualização (i.e.,`a`).

**DICAS**

+ Configure a função `gradientDescent` com os melhores valores para a taxa de decaimento ($k$) e passo de aprendizagem ($\alpha$) obtidos no item anterior.
+ Os parâmetros de entrada da função `gradientDescent` são descritos em seu cabeçalho. Veja a definição da função.
+ Treine o modelo com o conjunto de treinamento.
+ Configure o **número de épocas**, `n_epochs`, com o valor `2`, ou seja, o modelo será treinado por 2 épocas.
+ Lembre-se que a função hipótese é expressa no formato vetorial como $\hat{\textbf{y}}=\textbf{X}\textbf{a}$, onde $\textbf{X}$ é a matriz de atributos e $\textbf{a}$ é o vetor de pesos. Portanto, para fazer predições com as matrizes de atributos de treinamento e validação, você precisa utilizar a função hipótese no formato vetorial.
+ Você pode usar a função `mean_squared_error` da biblioteca SciKit-Learn para calcular o EQM.

In [ ]:
# Digite o código do exercício aqui.

8. Treine um modelo usando a **equação normal** (i.e., a equação que dá a solução ótima para o conjunto de treinamento fornecido). Ao final, imprima o erro quadrático médio (EQM) obtido pelo modelo para os conjuntos de treinamento e validação. Além disso, imprima o valor dos pesos $\hat{a}_0$ e $\hat{a}_1$ obtidos com a **equação normal**.

**DICAS**

+ Você pode utilizar a classe `LinearRegression` da biblioteca SciKit-Learn para resolver este item ou implementar a equação normal manualmente.
+ Caso você use a classe `LinearRegression`, a configure com o parâmetro `fit_intercept=False`, pois a matriz de atributos criada no item 2 do exercício, já contém a coluna do atributos de bias, ou seja, a coluna com todos os valores iguais a 1.
+ Usando a classe `LinearRegression`:
  * A predição é feita com o método `predict()`.
  * Os pesos do modelo podem ser acessados através do atributo `coef_` da classe `LinearRegression`. Por exemplo, dado que o nome do objeto da classe `LinearRegression` é `reg`, então `reg.coef_[0,0]` acessa o valor ótimo encontrado para o peso $\hat{a}_0$ e `reg.coef_[0,1]` acessa o valor ótimo encontrado para o peso $\hat{a}_1$.
+ Você pode usar a função `mean_squared_error` da biblioteca SciKit-Learn para calcular o EQM.

In [ ]:
# Digite o código do exercício aqui.

9. Compare os pesos ($\hat{a}_0$ e $\hat{a}_1$) e os erros, i.e., EQMs, (para os conjuntos de treinamento e validação) obtidos com os modelos usando a equação normal (item 8) e o gradiente descendente estocástico com os melhores valores para a taxa de decaimento e passo de aprendizagem (item 7). Em seguida, responda: os valores dos pesos são diferentes? Se sim, explique o motivo da diferença. (**Justifique sua resposta**).

**DICAS**

+ Lembre-se que a equação normal dá a solução ótima, ou seja, ela fornece os pesos que minimizam o EQM. Não existem outros pesos que resultem em um EQM menor para o conjunto de treinamento usado.
+ As estimativas do vetor gradiente com o gradiente descendente estocástico, mesmo com os melhores valores para a taxa de decaimento e passo de aprendizagem, continuam sendo ruidosas, consequentemente, as atualizações dos pesos também serão ruidosas.
+ Além disso, os valores encontrados para a taxa de decaimento e passo de aprendizagem podem não ser os ótimos.
+ Reveja o material de aula e os exemplos onde discutimos as versões do gradiente descendente.

<span style="color:blue">Digite aqui a resposta do exercício.</span>

**Resposta**

10. Plote a superfície de contorno desta função hipótese e mostre que os pesos encontrados com a equação normal e gradiente descendente são próximos, mas não idênticos. Para o gradiente descendente estocástico, use os pesos correspondentes à última iteração de atualização (i.e.,`a`).

**DICAS**

+ Use a função `calculateErrorSurface` definida no item 1 deste exercício.
+ A função `calculateErrorSurface` restringe o eixo de $\hat{a}_0$ entre os valores $0.995$ e $1.005$.
+ A função `calculateErrorSurface` restringe o eixo de $\hat{a}_1$ entre os valores $0.495$ e $0.505$.
+ Use as funções `xlim` e `ylim` da biblioteca matplotlib para restringir a figura aos limites de $\hat{a}_0$ e $\hat{a}_1$ mencionados acima ou a um limite menor, caso você prefira.

In [ ]:
# Digite o código do exercício aqui.

11. Plote uma figura que compare as funções objetivo, observável e aproximada (via gradiente descendente estocástico e equação normal). Use o conjunto total de amostras. No caso da função aproximada via gradiente descendente estocástico, use o vetor de pesos obtido com o `early-stopping` (i.e., `a_min`).

**DICAS**

+ Use a matriz $\textbf{X}$, criada no item 2, para fazer as predições com o conjunto total de dados.
+ Analise o código da função `gradientDescent` definida no item 1 para identificar qual é a variável com o vetor de pesos obtido com o `early-stopping`.

In [ ]:
# Digite o código do exercício aqui.

12. Discuta os resultados apresentados na figura anterior. O que você consegue concluir? (**Justifique sua resposta**)

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

# 2) **PROJETO SOBRE REGRESSÃO POLINOMIAL PARA MODELAGEM DE FDPs EM TELECOMUNICAÇÕES.**

**CONTEXTO: Em sistemas de telecomunicações modernos, como redes móveis (4G/5G/6G), sinais transmitidos sofrem diversos efeitos físicos, como ruído, interferência e propagação multipercurso (fading). Esses fenômenos tornam as variáveis de interesse, como potência do sinal ou qualidade do canal altamente complexas, fazendo com que suas funções densidade de probabilidade (FDPs) sejam desconhecidas ou não sigam distribuições simples (como a normal).**

Esse tipo de problema costuma aparecer também em outras áreas de telecom, como:

+ **Modelagem de canais de comunicação:** O sinal recebido pode seguir distribuições como Rayleigh ou Rician, ou até formas mais complexas devido à interferência agregada;

+ **Redes móveis (4G/5G/6G):** Métricas como RSSI (Received Signal Strength Indicator ou Indicador de Força do Sinal Recebido)  e SINR (Signal to Interference plus Noise Ratio ou Relação Sinal-Interferência mais Ruído)  possuem distribuições não triviais, muitas vezes multimodais, exigindo aproximações;

+ **Análise de tráfego de rede:** Variáveis como latência, throughput e volume de pacotes frequentemente apresentam distribuições com caudas longas ou comportamento bimodal;

+ **Detecção de sinais:** Decidir entre “sinal presente” ou “ausente” exige modelar distribuições de ruído e sinal + ruído, muitas vezes desconhecidas;

+ **Compressão e codificação:** Técnicas baseadas em entropia dependem de uma boa estimativa da FDP — quanto melhor a aproximação, maior a eficiência;

+ **Planejamento e otimização de redes:** A distribuição espacial de usuários e interferência pode gerar padrões multimodais, por exemplo, entre áreas urbanas e rurais.

Nestes cenários, nem sempre é possível obter uma expressão analítica da FDP. Portanto, utilizamos técnicas de aproximação de funções, como a regressão polinomial, para modelar o comportamento observado a partir de dados.

Sendo assim, neste projeto, você trabalhará com uma variável aleatória simulada que representa uma métrica típica de telecomunicações (como potência de sinal), cuja distribuição é bimodal e assimétrica, refletindo cenários reais, como:

+ Usuários próximos à estação base (sinal forte);

+ Usuários distantes ou com obstáculos (sinal fraco).

**Objetivo Principal:Encontre uma função que aproxime a FDP de uma variável aleatória desconhecida, utilizando regressão polinomial com base nos dados observados adiante.**

1. Execute a célula abaixo para observar a variável aleatória com FDP desconhecida gerada.

**Dicas:**

+ Sempre que possível, usem a semente (`seed`) definida na célula de código abaixo.
+ Este exercício consume muita memória RAM. Portanto, para que você não encontre problemas durante sua execução, use somente o Google Colab.

In [ ]:
# Importando todas as bibliotecas necessárias.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LassoCV

# Reset do gerador de sequências pseudo-aleatórias.
seed = 42
np.random.seed(seed)

# Número de amostras.
N = 10000000

# Distribuição Normal a.
mu_a = -1
sigma_a = 0.5

# Distribuição Normal b.
mu_b = 2
sigma_b = 1

# NOVA FDP: bimodal assimétrica (modificação feita aqui)
h1 = sigma_a * np.random.randn(int(0.7 * N)) + mu_a  # 70% dos dados
h2 = sigma_b * np.random.randn(int(0.3 * N)) + mu_b  # 30% dos dados

h = np.concatenate([h1, h2])

# Número de divisões do histograma.
bins = 1000
y, X, p = plt.hist(h, bins=bins, density=True)

plt.ylabel('Probabilidade')
plt.xlabel('y')
plt.title('FDP Bimodal Assimétrica')
plt.grid()
plt.show()

# Redimensionando o vetor de atributos.
X = X[0:len(X)-1].reshape(bins, 1)

# Imprimindo as dimensões.
print('X.shape:', X.shape)
print('y.shape:', y.shape)

2. Com base na matriz de atributos "X" e no vetor de rótulos "y" obtidos no item anterior, faça:

+ Utilize a técnica de validação cruzada k-Fold para escolher a melhor ordem para o modelo de aproximação da FDP;

+ Plote gráficos com a média e o desvio padrão do erro quadrático médio (MSE) em função dos graus de polinômio considerados.

Para isso:

 1. Use o **k-Fold** com **k** igual a 10 e o parâmetro `random_state=0`;
 2. Faça a análise de polinômios de ordem 1 até 50, **inclusive**;
 3. Não habilite a inclusão do atributo de bias ao instanciar a classe `PolynomialFeatures`. Desabilite a inclusão do atributo configurando o parâmetro como `include_bias=False`;
 4. Use a classe `StandardScaler` para escalonar/padronizar os dados.

**Dicas:**

+ Para resolver este item, se baseie no seguinte exemplo: [validacao_cruzada.ipynb](https://colab.research.google.com/github/zz4fap/t319_aprendizado_de_maquina/blob/main/notebooks/regression/validacao_cruzada.ipynb).

+ O tempo de execução dese exercício é longo. Portanto, pegue um café e tenha paciência.

+ **Atenção, não basta apenas copiar o código do exemplo dado, você precisa alterá-lo com base no código postado no link acima.**

In [ ]:
# Digite o código do exercício aqui.

3. Após analisar os resultados do item anterior, responda:

+ Qual a melhor ordem do polinômio para esse problema? **Justifique sua resposta.**

**Dicas:**

+ Use o princípio da navalha de Occam para escolher a ordem do polinômio.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


4. De posse da melhor ordem agora, você deverá treinar um novo modelo de regressão considerando esta ordem e, no final, imprima o valor do erro quadrático médio para (MSE).

**Dicas:**

+ Não habilite a inclusão do atributo de bias ao instanciar a classe `PolynomialFeatures`. Desabilite a inclusão do atributo configurando o parâmetro como `include_bias=False`;
+ Modifique o grau do polinômio em : `poly = PolynomialFeatures(degree=a)`, sendo "a" o  melhor valor do grau do polinômio encontrado por você;
+ Use a classe `StandardScaler` para padronizar os atributos;
+ Use o conjunto total de amostras para calcular o erro;
+ Para resolver este item, se baseie no seguinte exemplo: [validacao_cruzada.ipynb](https://colab.research.google.com/github/zz4fap/t319_aprendizado_de_maquina/blob/main/notebooks/regression/validacao_cruzada.ipynb).

+ **Atenção, não basta apenas copiar o código do exemplo dado, você precisa alterá-lo com base no código postado no link acima.**

In [ ]:
# Digite aqui o código do exercício.

5. Apresente uma figura comparando a predição realizada pelo melhor regressor com os dados originais.

**Dicas:**

+ Para resolver este item, se baseie no seguinte exemplo: [validacao_cruzada.ipynb](https://colab.research.google.com/github/zz4fap/t319_aprendizado_de_maquina/blob/main/notebooks/regression/validacao_cruzada.ipynb).

+ **Atenção, não basta apenas copiar o código do exemplo dado, você precisa alterá-lo com base no código postado no link acima.**

In [ ]:
# Digite aqui o código do exercício.

6. O quê aconteceria se a ordem do modelo fosse bem menor do que a que você escolheu (por exemplo, três vezes menor)? **Justifique sua resposta.**

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

7. Escolha uma ordem bem menor do que a que você usou no item 4 (por exemplo, 03 vezes menor) e apresente uma figura comparando a predição feita por esse modelo com ordem bem menor com os dados originais.

**Dicas:**

+ **Modifique o grau do polinômio em : `poly = PolynomialFeatures(degree=a, include_bias=False)` no item 4 para observar e responder com precisão a esta pergunta.**

In [ ]:
# Digite aqui o código do exercício.

8. Ao reduzir o grau do polinômio para 03 vezes menos que o grau do item 4, o MSE aumentou ou diminuiu em relação ao modelo ótimo?

O que esse variação no valor do erro indica sobre a capacidade do modelo e o fenômeno de underfitting/overfitting? (**Justifique sua resposta**)

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

# 3) **Exercício sobre regressão para a predição do preço de casas.**

Neste exercício, você irá desenvolver um modelo de **regressão linear** para prever o preço médio de imóveis em distritos da Califórnia.

O problema é clássico em Machine Learning e consiste em estimar o valor de casas com base em características (i.e., atributos) socioeconômicas e geográficas da região.

Esse tipo de aplicação é amplamente utilizado em:

* Avaliação imobiliária automatizada
* Sistemas de recomendação de investimento
* Planejamento urbano
* Análise socioeconômica

O dataset utilizado é o **California Housing**, derivado do censo dos EUA de 1990, contendo **20.640 amostras e 8 atributos numéricos** [1].

Cada amostra representa um **bloco censitário**, ou seja, uma pequena região geográfica com características demográficas homogêneas.

### **Descrição da Base de Dados**

### Atributos (Features)

| Atributo     | Tipo              | Descrição                               |
| ------------ | ----------------- | --------------------------------------- |
| `MedInc`     | Numérico contínuo | Renda mediana dos moradores do distrito |
| `HouseAge`   | Numérico contínuo | Idade mediana das casas                 |
| `AveRooms`   | Numérico contínuo | Número médio de cômodos por residência  |
| `AveBedrms`  | Numérico contínuo | Número médio de quartos                 |
| `Population` | Numérico contínuo | População total do distrito             |
| `AveOccup`   | Numérico contínuo | Número médio de pessoas por residência  |
| `Latitude`   | Numérico contínuo | Latitude geográfica                     |
| `Longitude`  | Numérico contínuo | Longitude geográfica                    |


### Variável alvo

| Nome          | Tipo              | Descrição                                                                            |
| ------------- | ----------------- | ------------------------------------------------------------------------------------ |
| `MedHouseVal` | Numérico contínuo | Valor médio das casas no distrito (em centenas de milhares de dólares) [2] |

Todos os atributos são numéricos e contínuos, o que torna o dataset ideal para regressão [3].


## **Objetivo do Exercício**

Desenvolver um modelo capaz de prever o preço médio das casas utilizando:

* Regressão Linear
* Validação cruzada
* Avaliação com métricas de regressão

### **Referências**:

[1] https://huggingface.co/datasets/gvlassis/california_housing?utm_source=chatgpt.com "gvlassis/california_housing · Datasets at Hugging Face"

[2] https://www.kaggle.com/datasets/shraddha4ever20/california-housing-dataset?utm_source=chatgpt.com "California Housing dataset"

[3] https://sequential-parameter-optimization.github.io/spotPython/reference/spotpython/data/california_housing/?utm_source=chatgpt.com "california_housing - spotpython"


1. Execute a célula abaixo para baixar a base de dados. Após a execução, as 10 primeiras linhas da base dedados são mostradas.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

# Defina uma semente a ser usada ao longo de todo o exercício.
seed = 42

# Carregar dataset
data = fetch_california_housing(as_frame=True)
df = data.frame

# Visualizar dados
df.head(10)

2. Execute o comando abaixo para obter uma visão geral da estrutura do conjunto de dados.

O método `df.info()` é utilizado para exibir um resumo conciso do DataFrame, fornecendo informações importantes sobre sua estrutura.

Ao executar esse comando, são apresentadas as seguintes informações:
+ Tipo do objeto (DataFrame do Pandas)
+ Número total de registros (linhas)
+ Intervalo do índice (ex: de 0 a 20639)
+ Quantidade de colunas
+ Nome de cada coluna
+ Número de valores não nulos em cada coluna
+ Tipo de dado (dtype) de cada atributo (ex: float64)
+ Uso de memória do DataFrame

Esse método é extremamente útil na fase inicial de análise exploratória dos dados, pois permite verificar rapidamente:
+ Se existem valores ausentes
+ Se os tipos de dados estão corretos
+ O tamanho do dataset

In [ ]:
# Informações gerais
df.info()

3. Com base no resultado do método `df.info()`, responda:

+ O dataset possui valores ausentes? Justifique sua resposta utilizando as informações apresentadas na saída do método.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


4. Execute o código abaixo para obter um resumo estatístico das variáveis numéricas do dataset.

O método `df.describe()` gera um conjunto de **estatísticas descritivas** que ajudam a entender a distribuição dos dados em cada coluna numérica do DataFrame.

Ele calcula automaticamente, para cada atributo:

* **count** → número de valores (não nulos)
* **mean** → média
* **std** → desvio padrão (dispersão dos dados)
* **min** → valor mínimo
* **25% (Q1)** → primeiro quartil
* **50% (mediana)** → valor central
* **75% (Q3)** → terceiro quartil
* **max** → valor máximo

Essas informações permitem analisar:

* A **distribuição dos dados**
* A presença de **valores extremos (outliers)**
* A **escala das variáveis**
* A variabilidade entre os atributos

**Interpretação do resultado apresentado**

A saída mostra uma tabela onde cada coluna corresponde a um atributo do dataset, e cada linha representa uma métrica estatística.



In [ ]:
df.describe()

5. Com base nos valores apresentados em `df.describe()`, analise o intervalo de variação dos atributos (`min`e `max`), verifique se eles são muito diferentes entre eles e responda se devemos aplicar escalonamento de atributos. (**Justifique sua resposta**)




**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


6. Execute o código abaixo para visualizar a relação entre as variáveis do dataset.

**Correlação**

O método `corr()` calcula a matriz de correlação entre todas as variáveis numéricas do DataFrame, incluindo atributos e rótulo.

Cada valor varia entre -1 e +1:
+ +1 → correlação positiva forte
+ 0 → ausência de relação linear
+ -1 → correlação negativa forte

Ou seja, ele mede o quanto duas variáveis variam juntas.

**Mapa de calor (heatmap)**

O `heatmap` transforma a matriz em um gráfico colorido.
A escala de cores indica a intensidade da correlação:
+ Vermelho → correlação positiva
+ Azul → correlação negativa

**Interpretação do resultado**

A saída é uma matriz simétrica, onde:
+ A diagonal principal sempre será 1 (variável correlacionada consigo mesma)
+ Valores mais próximos de ±1 indicam relações fortes
+ Valores próximos de 0 indicam pouca ou nenhuma relação linear

In [ ]:
# Plota a matriz de correlação
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Matriz de Correlação")
plt.show()

7. Com base na matriz de correlação, responda:

+ Qual variável apresenta maior influência linear sobre o preço das casas (MedHouseVal)?

Justifique sua resposta utilizando os valores observados no heatmap.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


8. Execute o código abaixo. Ele cria dois novos atributos no conjunto de dados e mostra as 5 primeiras linhas do conjunto.

A criação de novos atributos é chamada de engenharia de atributos e pode ajudar a melhorar o desempenho dos modelos de predição. A seguir, apresenta-se uma explicação sobre cada novo atributo criado.

+ **Rooms_per_Household** divide o número médio de cômodos por domicílio (AveRooms) pela média de ocupantes por domicílio (AveOccup). Isso revela quantos cômodos cada pessoa, em média, tem à sua disposição, eliminando o efeito do tamanho do agregado familiar.

+ **Bedrooms_per_Room** representa a proporção entre o número médio de quartos (AveBedrms) e o total de cômodos (AveRooms). Quanto menor essa razão, mais cômodos não destinados a dormitórios (salas, cozinhas, etc.) existem em relação ao número de quartos, o que pode indicar imóveis maiores ou com mais áreas sociais.

+ **Population_per_Household** divide a população total do distrito (`Population`) pelo número médio de ocupantes por domicílio (`AveOccup`). O resultado é uma estimativa do número de domicílios no distrito, permitindo comparar o tamanho populacional independentemente da lotação média das residências.

+ **MedInc_per_Capita** resulta da divisão da renda mediana do distrito (`MedInc`) pelo número médio de ocupantes por domicílio (`AveOccup`). Essa razão aproxima a renda mediana por pessoa, ajustando o poder aquisitivo ao tamanho do agregado familiar e removendo o efeito de famílias numerosas sobre a métrica original de renda domiciliar.

+ **PopDensityIndex** é obtido calculando-se `Population / (AveRooms * AveOccup)`. A expressão `AveRooms * AveOccup` estima a quantidade total de cômodos no distrito, portanto o índice representa quantos habitantes existem, em média, para cada cômodo disponível. Valores altos indicam maior adensamento residencial, sinalizando imóveis mais ocupados e possivelmente menos valorizados.

+ **Rooms_per_Bedroom** é a divisão do número médio de cômodos por domicílio (`AveRooms`) pelo número médio de quartos (`AveBedrms`). Essa proporção revela quantos cômodos (salas, cozinhas, banheiros, etc.) existem para cada quarto no domicílio típico. Quanto maior o valor, mais amplo e diversificado tende a ser o imóvel, sugerindo plantas maiores ou com mais áreas de convivência.

**OBS**.: Observe que no código abaixo nós guardamos uma cópia do dataset sem engenharia de atributos para uso futuro.

In [ ]:
# Guardamos uma cópia do dataset sem engenharia de atributos.
df_no_feature_eng = df.copy()

# Criando as novas colunas
df['Rooms_per_Household'] = df['AveRooms'] / df['AveOccup']
df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']
df['Population_per_Household'] = df['Population'] / df['AveOccup']
df['MedInc_per_Capita'] = df['MedInc'] / df['AveOccup']
df['PopDensityIndex'] = df['Population'] / (df['AveRooms'] * df['AveOccup'])
df['Rooms_per_Bedroom']  = df['AveRooms'] / df['AveBedrms']

df.head()

9. Execute o código abaixo para visualizar a relação entre as variáveis do dataset. Preste atenção às correlações entre os novos atributos e a variável alvo (`MedHouseVal`).

In [ ]:
# As próximas duas linhas são usadas para manter a variável alvo ('MedHouseVal') como a última coluna do dataset.
col = df.pop('MedHouseVal')
df['MedHouseVal'] = col

# Plota a matriz de correlação
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Matriz de Correlação")
plt.show()

10. Com base na matriz de correlação acima, responda:

+ Os novos atributos apresentam boa correlação linear com o preço das casas (MedHouseVal)?
+ Quais dos novos atributos têm alta correlação com a variável alvo (MedHouseVal)?
+ Essas correlações são maiores do que as que existiam com os atributos originais.

Justifique sua resposta utilizando os valores observados no heatmap.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


11. Execute a célula abaixo para criar a matriz de atributos e o vetor de rótulos.

In [ ]:
# Cria um novo DataFrame removendo a coluna MedHouseVal, ou seja, mantém apenas as variáveis de entrada (X).
# O método drop() é usado para remover a coluna 'MedHouseVal' do DataFrame.
X = df.drop(columns=['MedHouseVal'])

# Seleciona apenas a coluna MedHouseVal, que será a variável alvo (y), ou seja, aquilo que o modelo deve prever
y = df['MedHouseVal']

12. Divida o conjunto total de amostras em 80% para treinamento e 20% para teste/validação.

In [ ]:
# Digite o código do exercício aqui.

13. Padronize os conjuntos de dados de treinamento e de teste/validação.

In [ ]:
# Digite o código do exercício aqui.

14. Instancie e treine um modelo de regressão linear. Apresente o erro quadrático médico (EQM) calculado para os conjuntos de treinamento e teste/validação.

In [ ]:
# Digite o código do exercício aqui.

15. Execute a célula abaixo. Ela utiliza o conjunto de dados **sem os novos atributos** criados usando engenharia de atributos.

O código separa o conjunto em atributos e rótulos, cria os subconjuntos de treinamento e teste/validação, padroniza os atributos, treina e avalia um modelo de regressão linear.

In [ ]:
# Cria um novo DataFrame removendo a coluna MedHouseVal, ou seja, mantém apenas as variáveis de entrada (X).
# O método drop() é usado para remover a coluna 'MedHouseVal' do DataFrame.
X_nfe = df_no_feature_eng.drop(columns=['MedHouseVal'])

# Seleciona apenas a coluna MedHouseVal, que será a variável alvo (y), ou seja, aquilo que o modelo deve prever
y_nfe = df_no_feature_eng['MedHouseVal']

# Divisão em conjuntos de treinamento e teste/validação.
X_train_nfe, X_test_nfe, y_train_nfe, y_test_nfe = train_test_split(
    X_nfe, y_nfe, test_size=0.2, random_state=seed
)

# Padronização
scaler_nfe = StandardScaler()
X_train_scaled_nfe = scaler_nfe.fit_transform(X_train_nfe)
X_test_scaled_nfe = scaler_nfe.transform(X_test_nfe)

# Treina e avalia um modelo de regressão linear.
model_nfe = LinearRegression()
model_nfe.fit(X_train_nfe, y_train_nfe)

y_pred_nfe = model_nfe.predict(X_train_nfe)
print("MSE train:", mean_squared_error(y_train_nfe, y_pred_nfe))

y_pred_nfe = model_nfe.predict(X_test_nfe)
print("MSE test:", mean_squared_error(y_test_nfe, y_pred_nfe))

16. Compare os EQMs de treinamento e teste/validação obtidos com os modelos treinados com e sem engenharia de atributos e responda:

+ A engenharia de atributos impactou positivamente o modelo treinado com novos atributos? (**Justifique sua resposta**).

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>
